# Distinctions Principle: Theory to Operationalization

This notebook encodes a persistent distinction constraint and compares constrained vs unconstrained dynamics.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from experiments.core import Distinction, DistinctionSet
from experiments.models.baseline import BaselineComplexSystemModel
from experiments.models import run_simulation


In [ ]:
distinction = Distinction(
    name="feature-0-invariant",
    description="Keep feature 0 near 0.5 for most agents",
    parameters={
        "feature_index": 0,
        "target_value": 0.5,
        "tolerance": 0.15,
        "min_fraction": 0.6,
    },
)

distinctions = DistinctionSet([distinction])
distinctions


In [ ]:
base_kwargs = dict(n_agents=20, n_features=3, interaction_strength=0.3, seed=42)
steps = 100

baseline_model = BaselineComplexSystemModel(**base_kwargs)
constrained_model = BaselineComplexSystemModel(**base_kwargs)

baseline = run_simulation(baseline_model, steps=steps, distinctions=DistinctionSet([]))
constrained = run_simulation(constrained_model, steps=steps, distinctions=distinctions)

baseline_traj = np.array([state.state for state in baseline.trajectory])
constrained_traj = np.array([state.state for state in constrained.trajectory])


In [ ]:
feature_index = distinction.parameters["feature_index"]
target = distinction.parameters["target_value"]
tolerance = distinction.parameters["tolerance"]

baseline_feature = baseline_traj[:, feature_index::base_kwargs["n_features"]]
constrained_feature = constrained_traj[:, feature_index::base_kwargs["n_features"]]

baseline_in_band = (np.abs(baseline_feature - target) <= tolerance).mean(axis=1)
constrained_in_band = (np.abs(constrained_feature - target) <= tolerance).mean(axis=1)

print("Average compliance (baseline):", round(float(baseline_in_band.mean()), 3))
print("Average compliance (constrained):", round(float(constrained_in_band.mean()), 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(baseline_feature.mean(axis=1), label="Baseline")
axes[0].plot(constrained_feature.mean(axis=1), label="Constrained")
axes[0].axhline(target, linestyle="--", color="red", alpha=0.6, label="Target")
axes[0].set_title("Feature Mean Over Time")
axes[0].set_xlabel("Step")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(baseline_in_band, label="Baseline")
axes[1].plot(constrained_in_band, label="Constrained")
axes[1].axhline(distinction.parameters["min_fraction"], linestyle="--", color="red", alpha=0.6, label="Min fraction")
axes[1].set_ylim(0, 1)
axes[1].set_title("Fraction of Agents in Tolerance Band")
axes[1].set_xlabel("Step")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
